[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/14_Ontology.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 本体生成（Ontology Generation）

欢迎使用 Semantica 本体模块的全面指南。这个模块是把数据结构化成有意义知识图谱的动力源——提供一条从原始数据到经过校验的 OWL 本体的完整流水线。

（注：原文开头称"6 阶段"流水线，正文标题又写"5 阶段"，实际列出的阶段是 5 个——译文按 5 个阶段叙述。）

本 notebook 将深入：

1. **生成流水线**：理解 Semantica 如何把数据变成知识
2. **核心组件细讲**：`ClassInferrer`、`PropertyGenerator`、`OntologyOptimizer` 的详细用法
3. **可视化**：用交互式图表和层级树探索你的本体
4. **高级用法**：文本生成本体（LLM）、能力问题、生命周期管理
5. **导出与互操作**：以 Turtle、RDF/XML 等标准格式保存成果

**文档**：[API 参考](https://semantica.readthedocs.io/reference/ontology/)

### 先看主线

图谱和本体是什么关系？**图谱记录"有什么"，本体规定"能有什么"**。打个比方：图谱是城市里一栋栋实际的楼，本体是城市规划法规——哪类地块能建什么、什么性质的设施不能相邻。07–10 篇建的是"楼"，这一篇立的是"法"。

本篇按"一件本体的一生"五段展开：

**生出来 —— 一条流水线**

`OntologyEngine.from_data` 一发入魂，原始数据直接变本体。

**调得细 —— 三个组件**

自动挡跑完想手动调？`ClassInferrer` 发现类、`PropertyGenerator` 生成属性、`OntologyOptimizer` 去冗提质。

**看得见 —— 可视化**

层级树、结构网络、指标盘。

**活得久 —— 生命周期与 AI**

LLM 从文本直接生成本体、用"能力问题"给本体做单元测试、版本管理与标准复用。

**带得走 —— 导出**

Turtle、RDF/XML 标准格式，语义网工具通用。

## 开始

先设置环境、初始化 `OntologyEngine`。这个引擎是所有本体操作的统一入口。

In [ ]:
!pip install -q semantica

In [ ]:

from semantica.ontology import OntologyEngine, OntologyGenerator
from semantica.utils.logging import get_logger

# Initialize logger for visibility
logger = get_logger("ontology_guide")

# Initialize the Engine
# base_uri defines the namespace root for your ontology
engine = OntologyEngine(base_uri="https://docs.semantica.dev/ontology/", min_occurrences=1)

print("Ontology Engine initialized successfully!")

---

## 生成流水线

Semantica 用一条多阶段流水线稳健地生成本体：吃进原始的实体和关系数据，产出高质量的 OWL 本体。

### 各阶段：

1. **语义网络解析**：从输入中提取原始概念与连接
2. **YAML 转定义**：把概念变换成结构化的类定义
3. **定义转类型**：把定义映射成正式的 OWL 类型（如 `owl:Class`、`owl:ObjectProperty`）
4. **层级生成**：用 `associatedWith` 或语言模式构建分类结构（父子关系）
5. **TTL 生成**：把内存结构序列化成 Turtle 格式

用示例数据跑一遍看效果。

In [ ]:
# Sample Data: A simple corporate structure
entities = [
    {"id": "e1", "type": "Company", "name": "TechCorp", "founded": "2010"},
    {"id": "e2", "type": "Person", "name": "Alice", "role": "CEO"},
    {"id": "e3", "type": "Person", "name": "Bob", "role": "CTO"},
    {"id": "e4", "type": "Department", "name": "Engineering"},
    {"id": "e5", "type": "Project", "name": "Project Phoenix"},
    # Added entities to ensure robust class inference
    {"id": "e6", "type": "Company", "name": "TechSolutions", "founded": "2015"},
    {"id": "e7", "type": "Person", "name": "Charlie", "role": "Lead"},
    {"id": "e8", "type": "Person", "name": "Dave", "role": "Manager"},
    {"id": "e9", "type": "Department", "name": "Research"},
    {"id": "e10", "type": "Project", "name": "Project Orion"}
]

relationships = [
    {"source": "e2", "target": "e1", "type": "leads"},
    {"source": "e3", "target": "e4", "type": "manages"},
    {"source": "e4", "target": "e1", "type": "part_of"},
    {"source": "e3", "target": "e5", "type": "works_on"},
    # Additional relationships for new entities
    {"source": "e7", "target": "e6", "type": "leads"},
    {"source": "e8", "target": "e9", "type": "manages"},
    {"source": "e9", "target": "e6", "type": "part_of"},
    {"source": "e7", "target": "e10", "type": "works_on"}
]

data = {
    "entities": entities,
    "relationships": relationships
}

# Run the full pipeline
ontology = engine.from_data(data, name="CorporateOntology")

print(f"Generated Ontology: {ontology['name']}")
print(f"Classes Found: {len(ontology['classes'])}")
print(f"Properties Found: {len(ontology['properties'])}")

### 检查产出

生成的 `ontology` 对象是个内容丰富的字典，装着全部推断出的结构。看看里面创建了哪些类和属性。

In [ ]:
# Inspect Classes
print("--- Inferred Classes ---")
for cls in ontology['classes']:
    print(f"Class: {cls['name']}")
    print(f"  URI: {cls.get('uri')}")
    # Check if a hierarchy was inferred
    if cls.get('subClassOf'):
        print(f"  Parent: {cls['subClassOf']}")
    print("")

# Inspect Properties
print("--- Inferred Properties ---")
for prop in ontology['properties']:
    type_label = "Object Property" if prop['type'] == 'object' else "Data Property"
    print(f"{prop['name']} [{type_label}]")
    print(f"  Domain: {prop.get('domain')}")
    print(f"  Range: {prop.get('range')}")
    print("")

---

## 深入：逐个组件看

`OntologyEngine` 适合一键生成，但精细控制常常免不了。看看引擎背后的单件工具——相当于从自动挡换回手动挡。

### 1. `ClassInferrer`：类发现

`ClassInferrer` 分析实体找模式。它能容忍噪声，只为出现足够频繁的类型建类。

*   **`min_occurrences`**：少于这个实体数的类型直接忽略
*   **`build_class_hierarchy`**：开关自动的父子检测

In [ ]:
from semantica.ontology import ClassInferrer

# Initialize inferrer with a threshold
# We set min_occurrences=1 here to capture everything in our small example
inferrer = ClassInferrer(min_occurrences=1)

raw_entities = [
    {"type": "Manager", "name": "Dave", "level": 5},
    {"type": "Manager", "name": "Eve", "level": 4},
    {"type": "Employee", "name": "Frank"}, # Only 1 employee
    {"type": "TemporaryWorker", "name": "Grace"} 
]

# Infer classes
classes = inferrer.infer_classes(raw_entities, build_hierarchy=True)

print(f"Inferred {len(classes)} classes from raw entities.")
for c in classes:
    print(f"- {c['name']} (Count: {c['entity_count']})")

### 2. `PropertyGenerator`：本体的粘合剂

属性定义关系。Semantica 区分两类：

*   **对象属性（Object Property）**：两个实体之间的链接（如 Person 与 Company 之间的 `leads`）
*   **数据属性（Data Property）**：实体的属性值（如 Company 的 `founded` 年份）

`PropertyGenerator` 自动识别这个区别。

In [ ]:
from semantica.ontology import PropertyGenerator

prop_gen = PropertyGenerator()

# We need the classes first to help property generation context
context_classes = classes  # reusing from previous step

# Let's define some relationships and attributes implicitly via entities
# Note: 'level' in Manager entities is a potential data property
complex_entities = [
    {"id": "m1", "type": "Manager", "name": "Dave", "level": 5},
    {"id": "e1", "type": "Employee", "name": "Frank"}
]
complex_relationships = [
    {"source": "m1", "target": "e1", "type": "supervises"} # Object property
]

properties = prop_gen.infer_properties(
    entities=complex_entities,
    relationships=complex_relationships,
    classes=context_classes
)

print("--- Property Types Identified ---")
for p in properties:
    print(f"Property: {p['name']}")
    print(f"  Type: {p['type']}")
    print(f"  Domain: {p['domain']} -> Range: {p['range']}")

### 3. `OntologyOptimizer`：精炼结构

定稿前最好优化一遍。优化器去除冗余、提升一致性——比如确保所有类都有合规的标签和有效的 URI。

In [ ]:
from semantica.ontology import OntologyOptimizer

optimizer = OntologyOptimizer()

# Let's pretend we have a messy ontology dict
messy_ontology = {
    "classes": [
        {"name": "Person", "uri": "...Person"},
        {"name": "Person", "uri": "...Person"} # Duplicate!
    ],
    "properties": []
}

clean_ontology = optimizer.optimize_ontology(messy_ontology, remove_redundancy=True)

print(f"Original Classes: {len(messy_ontology['classes'])}")
print(f"Optimized Classes: {len(clean_ontology['classes'])}")

---

## 可视化

一张图胜过一千个三元组！`OntologyVisualizer` 让你交互式地探索本体结构。

可以画：

*   **类层级**：类继承的树状图
*   **结构网络**：类与属性的全景图
*   **指标面板**：一眼看全的统计概览

In [ ]:
from semantica.visualization import OntologyVisualizer

viz = OntologyVisualizer()

# 1. Interactive Class Hierarchy
# Returns a Plotly figure you can interact with
fig_hierarchy = viz.visualize_hierarchy(ontology, output="interactive")
if fig_hierarchy:
    fig_hierarchy.show()

# 2. Ontology Structure Network
# See how classes and properties connect
fig_structure = viz.visualize_structure(ontology, output="interactive")
if fig_structure:
    fig_structure.show()

# 3. Metrics Dashboard
# View counts, depths, and statistics
fig_metrics = viz.visualize_metrics(ontology, output="interactive")
if fig_metrics:
    fig_metrics.show()

---

## 高级用法：生命周期与 AI

企业本体是有生命的东西，要长大、要换代。Semantica 提供管理完整生命周期的工具，也提供 AI 加速创建。

### 1. 文本生成本体（LLM 集成）

不想手工建实体？用 `LLMOntologyGenerator` 直接从文本需求或文档里抽出一个本体。

In [ ]:
from semantica.ontology import LLMOntologyGenerator

try:
    # Note: Requires an API key in your environment variables
    llm_gen = LLMOntologyGenerator(provider="openai", model="gpt-4")

    text_description = """
    A University has many Departments. Each Department offers several Courses.
    Professors teach Courses and belong to a Department.
    Students enroll in Courses.
    """

    llm_ontology = llm_gen.generate_ontology_from_text(
        text=text_description,
        name="UniversityOntology"
    )

    print("Generated Classes:", [c['name'] for c in llm_ontology['classes']])
except Exception:
    print("Skipping LLM generation: No API key or provider configured in this environment.")

### 2. 测试驱动设计（能力问题）

把需求形式化成"能力问题"（Competency Questions，CQ）。`CompetencyQuestionsManager` 能检查你的本体是否包含回答这些问题所需的术语——相当于给本体写单元测试。

In [ ]:
from semantica.ontology import CompetencyQuestionsManager

cq_manager = CompetencyQuestionsManager()

# Define what our ontology SHOULD answer
cq_manager.add_question("Who leads TechCorp?", category="organizational")
cq_manager.add_question("Which projects does Bob manage?", category="operational")

# Validate our 'ontology' against these questions
validation_results = cq_manager.validate_ontology(ontology)

print(f"Answerable Questions: {validation_results['answerable']} / {validation_results['total_questions']}")
for q in cq_manager.questions:
    status = "✅" if q.answerable else "❌"
    print(f"{status} {q.question}")

### 3. 生命周期管理（版本与复用）

用 `VersionManager` 管理版本迭代，用 `ReuseManager` 导入 FOAF、Dublin Core 这类外部标准——成熟的轮子不必重造。

In [ ]:
from semantica.ontology import VersionManager, ReuseManager

# --- Versioning ---
v_manager = VersionManager(base_uri="https://example.org/ontology/")
v1 = v_manager.create_version("1.0", ontology, changes=["Initial creation"])
print(f"Created Version: {v1.version} at {v1.ontology_iri}")

# --- Reuse ---
reuse_manager = ReuseManager()

# Check if we can reuse FOAF
foaf_info = reuse_manager.research_ontology("http://xmlns.com/foaf/0.1/")
if foaf_info:
    print(f"Found standard ontology: {foaf_info['name']}")
    # We could now import this into our ontology
    ontology['imports'].append(foaf_info['uri'])

---

## 导出本体

本体建好、验证完，就该保存了。Semantica 以 **Turtle（`.ttl`）**为主格式，也经 `rdflib` 支持其他格式。

可以导出成字符串，也可以直接写文件。

In [ ]:
# Get Turtle string representation
ttl_output = engine.to_owl(ontology, format="turtle")

print("--- Turtle Preview (First 500 chars) ---")
print(ttl_output[:500])
print("...")

# Save to file
output_path = "corporate_ontology.ttl"
engine.export_owl(ontology, path=output_path, format="turtle")
print(f"Successfully saved ontology to {output_path}")

## 小结

### 回顾主线

一件本体的一生走完：**生出来**（流水线从数据到结构）、**调得细**（三个组件逐件打磨）、**看得见**（三类交互图）、**活得久**（LLM 生成、能力问题测试、版本与复用）、**带得走**（Turtle/RDF 标准导出）。记住开头的比方：图谱是楼，本体是法——楼和法都有了，后面的篇章会把两者合起来用。

### 你已掌握

*   **自动生成**：用多阶段流水线从原始数据到结构化本体
*   **组件控制**：用 `ClassInferrer` 和 `PropertyGenerator` 精细建模
*   **可视化**：交互式探索本体结构
*   **高级生命周期**：AI 生成、能力问题、版本管理
*   **导出**：序列化成果，供其他语义网工具使用

### 下一步

*   试试定制 `NamespaceManager`，用你所在组织的 URL
*   探索 `OntologyEvaluator` 获取更深入的质量指标
*   把生成的本体喂给**知识图谱**模块，开始对你的数据做推理
*   把图、本体和显式映射放在一起，见 [Semantic Layer Basics](./26_Semantic_Layer_Basics.ipynb)